In [3]:
# load_data.py  (second file)
from pathlib import Path
import tensorflow as tf

WORK_DIR = Path("animal")   # must match the first file
IMG_SIZE = (64, 64)
SEED     = 42
BATCH    = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    str(WORK_DIR / "train"), image_size=IMG_SIZE, batch_size=BATCH,
    seed=SEED, shuffle=True)
val_ds = tf.keras.utils.image_dataset_from_directory(
    str(WORK_DIR / "val"), image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)
test_ds = tf.keras.utils.image_dataset_from_directory(
    str(WORK_DIR / "test"), image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)

print("Classes:", train_ds.class_names)

# Scale pixels to 0-1 for the custom model A
normalize = tf.keras.layers.Rescaling(1.0 / 255)
train_ds_n = train_ds.map(lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE)
val_ds_n   = val_ds.map(lambda x, y: (normalize(x), y),   num_parallel_calls=tf.data.AUTOTUNE)
test_ds_n  = test_ds.map(lambda x, y: (normalize(x), y),  num_parallel_calls=tf.data.AUTOTUNE)

Found 2100 files belonging to 3 classes.
Found 450 files belonging to 3 classes.
Found 450 files belonging to 3 classes.
Classes: ['cats', 'dogs', 'snakes']


In [4]:
from tensorflow.keras import layers, models
import time

tf.keras.utils.set_random_seed(SEED)        # reproducible weight initialisation
NUM_CLASSES = len(train_ds.class_names)
AUTOTUNE = tf.data.AUTOTUNE

# Keep decoded images in memory and prepare the next batch while the GPU trains (faster epochs).
# Training data: cache single images, then re-shuffle every epoch and re-batch.
train_ds_n = (train_ds_n.unbatch().cache()
              .shuffle(2000, seed=SEED, reshuffle_each_iteration=True)
              .batch(BATCH).prefetch(AUTOTUNE))
val_ds_n   = val_ds_n.cache().prefetch(AUTOTUNE)
test_ds_n  = test_ds_n.cache().prefetch(AUTOTUNE)

def build_model_a(input_shape=(64, 64, 3), num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape)
    x = inputs
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, kernel_size=3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inputs, outputs, name="Model_A_StandardCNN")

model_a = build_model_a()